# RAG Specification Sensitivity Demo

Two RAG chatbots over the AtlasED corpus — one using the full England corpus (3,939 articles including Schools Week), one using the NM corpus (1,197 articles without Schools Week). Same question, different corpus, different answer.

**Architecture:** Sentence-transformer embeddings → FAISS vector store → LangGraph agent → Claude → Langfuse logging

**The specification sensitivity point:** The RAG chatbot's answers change depending on which documents are in the retrieval corpus. This is the same finding as the NMF comparison — source composition determines what the system finds — but applied to the architecture that matters in 2026.

## 0. Imports and setup

In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("/workspaces/AM1_topic_modelling/.env"))

from sentence_transformers import SentenceTransformer
import faiss
from anthropic import Anthropic

print("All imports loaded.")

/workspaces/AM1_topic_modelling/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports loaded.


## 1. Load and prepare both corpora

In [2]:
# Full corpus (with Schools Week)
full_df = pd.read_csv("/workspaces/AM1_topic_modelling/data/evaluation_outputs/topics_analysis_ready_eng.csv")
full_df["article_date"] = pd.to_datetime(full_df["article_date"], errors="coerce")
full_df["chunk_id"] = range(len(full_df))
full_df["text_for_embed"] = full_df["text_clean"].fillna("").str[:1500]  # truncate for embedding

# NM corpus (without Schools Week)
nm_df = full_df[full_df["source"] != "schoolsweek"].copy().reset_index(drop=True)
nm_df["chunk_id"] = range(len(nm_df))

print(f"Full corpus: {len(full_df)} articles ({full_df['source'].value_counts().to_dict()})")
print(f"NM corpus:   {len(nm_df)} articles ({nm_df['source'].value_counts().to_dict()})")

Full corpus: 3939 articles ({'schoolsweek': 2741, 'gov': 679, 'fft': 202, 'epi': 111, 'nuffield': 106, 'fed': 100})
NM corpus:   1198 articles ({'gov': 679, 'fft': 202, 'epi': 111, 'nuffield': 106, 'fed': 100})


## 2. Embed both corpora with sentence-transformers

In [3]:
# Use MiniLM — small, fast, CPU-friendly (itself a specification choice)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding model: all-MiniLM-L6-v2 (dim={embed_model.get_sentence_embedding_dimension()})")

# Embed full corpus
print("Embedding full corpus...")
full_embeddings = embed_model.encode(
    full_df["text_for_embed"].tolist(),
    show_progress_bar=True,
    batch_size=64,
)
print(f"Full embeddings shape: {full_embeddings.shape}")

# Embed NM corpus
print("Embedding NM corpus...")
nm_embeddings = embed_model.encode(
    nm_df["text_for_embed"].tolist(),
    show_progress_bar=True,
    batch_size=64,
)
print(f"NM embeddings shape: {nm_embeddings.shape}")

Embedding model: all-MiniLM-L6-v2 (dim=384)
Embedding full corpus...


Batches: 100%|█████████████████████████████████████████████████| 62/62 [03:31<00:00,  3.40s/it]


Full embeddings shape: (3939, 384)
Embedding NM corpus...


Batches: 100%|█████████████████████████████████████████████████| 19/19 [00:53<00:00,  2.81s/it]

NM embeddings shape: (1198, 384)


## 3. Build FAISS indices

In [4]:
dim = full_embeddings.shape[1]

# Full corpus index
full_index = faiss.IndexFlatIP(dim)  # inner product (cosine similarity on normalised vectors)
faiss.normalize_L2(full_embeddings)
full_index.add(full_embeddings)
print(f"Full FAISS index: {full_index.ntotal} vectors")

# NM corpus index
nm_index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(nm_embeddings)
nm_index.add(nm_embeddings)
print(f"NM FAISS index: {nm_index.ntotal} vectors")

# Save indices for reuse
output_dir = Path("/workspaces/AM1_topic_modelling/data/rag")
output_dir.mkdir(exist_ok=True)
faiss.write_index(full_index, str(output_dir / "full_corpus.faiss"))
faiss.write_index(nm_index, str(output_dir / "nm_corpus.faiss"))
print(f"Saved to {output_dir}")

Full FAISS index: 3939 vectors
NM FAISS index: 1198 vectors
Saved to /workspaces/AM1_topic_modelling/data/rag


## 4. Retrieval function

In [5]:
def retrieve(query, index, corpus_df, embed_model, k=10):
    """Retrieve top-k articles from FAISS index for a given query."""
    query_vec = embed_model.encode([query])
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = corpus_df.iloc[idx]
        results.append({
            "text": str(row["text_clean"])[:1000],
            "source": row["source"],
            "date": str(row["article_date"])[:10],
            "topic": row["topic_name"],
            "score": float(score),
        })
    return results

# Test retrieval
test_q = "What are the main issues with AI in education?"
full_results = retrieve(test_q, full_index, full_df, embed_model)
nm_results = retrieve(test_q, nm_index, nm_df, embed_model)

print(f"Query: {test_q}")
print(f"\nFull corpus — top 5 sources: {[r['source'] for r in full_results[:5]]}")
print(f"NM corpus   — top 5 sources: {[r['source'] for r in nm_results[:5]]}")

Query: What are the main issues with AI in education?

Full corpus — top 5 sources: ['schoolsweek', 'fed', 'nuffield', 'schoolsweek', 'schoolsweek']
NM corpus   — top 5 sources: ['fed', 'nuffield', 'fed', 'gov', 'fed']


## 5. RAG answer function with Claude

In [6]:
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def rag_answer(question, index, corpus_df, embed_model, corpus_label, k=10):
    """Retrieve articles and generate an answer using Claude."""
    
    # Retrieve
    results = retrieve(question, index, corpus_df, embed_model, k=k)
    
    # Build context
    context_parts = []
    for i, r in enumerate(results):
        context_parts.append(
            f"ARTICLE {i+1} (source: {r['source']}, date: {r['date']}, topic: {r['topic']}):\n{r['text']}"
        )
    context = "\n\n".join(context_parts)
    
    # Generate answer
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[{"role": "user", "content": f"""You are an education policy analyst. Based on the following articles from the {corpus_label} corpus, answer this question:

Question: {question}

{context}

Answer based only on the articles provided. Cite which sources and dates support your points. Be specific about what the evidence shows."""}],
    )
    
    answer = response.content[0].text
    sources_used = [r["source"] for r in results]
    topics_covered = list(set(r["topic"] for r in results))
    
    return {
        "answer": answer,
        "sources_used": sources_used,
        "topics_covered": topics_covered,
        "n_retrieved": len(results),
        "corpus": corpus_label,
    }

print("RAG answer function ready.")

RAG answer function ready.


## 6. Side-by-side comparison — same question, different corpus

In [7]:
def compare_corpora(question):
    """Ask the same question to both corpora and compare answers."""
    
    print(f"QUESTION: {question}")
    print("=" * 80)
    
    # Full corpus
    print("\n[Querying full corpus (with Schools Week)...]")
    full_answer = rag_answer(question, full_index, full_df, embed_model, "full England corpus (including Schools Week)")
    
    # NM corpus
    print("[Querying NM corpus (without Schools Week)...]")
    nm_answer = rag_answer(question, nm_index, nm_df, embed_model, "England corpus without media (Schools Week removed)")
    
    # Display
    print("\n" + "=" * 80)
    print("FULL CORPUS ANSWER (with Schools Week):")
    print("-" * 40)
    print(full_answer["answer"])
    print(f"\nSources retrieved: {pd.Series(full_answer['sources_used']).value_counts().to_dict()}")
    print(f"Topics covered: {full_answer['topics_covered']}")
    
    print("\n" + "=" * 80)
    print("NM CORPUS ANSWER (without Schools Week):")
    print("-" * 40)
    print(nm_answer["answer"])
    print(f"\nSources retrieved: {pd.Series(nm_answer['sources_used']).value_counts().to_dict()}")
    print(f"Topics covered: {nm_answer['topics_covered']}")
    
    print("\n" + "=" * 80)
    print("SPECIFICATION SENSITIVITY:")
    print(f"  Full corpus sources: {pd.Series(full_answer['sources_used']).value_counts().to_dict()}")
    print(f"  NM corpus sources:   {pd.Series(nm_answer['sources_used']).value_counts().to_dict()}")
    print(f"  Same question, different corpus → different answer.")
    
    return full_answer, nm_answer

In [8]:
# Question 1: broad policy landscape
q1_full, q1_nm = compare_corpora("What are the main issues in English education policy?")

QUESTION: What are the main issues in English education policy?

[Querying full corpus (with Schools Week)...]
[Querying NM corpus (without Schools Week)...]

FULL CORPUS ANSWER (with Schools Week):
----------------------------------------
Based on the articles provided, the main issues in English education policy are:

## School Funding Crisis
**School funding emerges as the top concern among education professionals.** According to a National Foundation for Educational Research survey reported in Schools Week (June 3, 2024), school funding was identified as the most important education issue by teachers and leaders, with "per cent" placing it in their top three priorities ahead of the general election.

## Teacher Recruitment and Retention Crisis
**There is a chronic staffing crisis across the education system.** The Education Policy Institute report (December 1, 2023) identifies that "education at every level is experiencing a chronic recruitment and retention challenge." This is rei

In [9]:
# Question 2: specific topic
q2_full, q2_nm = compare_corpora("What is the government doing about AI in schools?")

QUESTION: What is the government doing about AI in schools?

[Querying full corpus (with Schools Week)...]
[Querying NM corpus (without Schools Week)...]

FULL CORPUS ANSWER (with Schools Week):
----------------------------------------
Based on the articles provided, the government is taking several concrete actions regarding AI in schools:

## Policy Development and Guidance

The government has developed comprehensive AI guidance for schools. According to a June 10, 2025 gov.uk article, the Department for Education "launched a package of measures to transform how schools use ai including the first ever ai guidance for schools and colleges setting out how schools can safely and effectively use ai to transform the classroom for students."

## Strategic Planning and Investment

The government has established a new "digital centre of government" within the Department for Science, Information and Technology to focus on AI deployment in public services, including education (Schools Week, Ja

In [10]:
# Question 3: contested topic
q3_full, q3_nm = compare_corpora("What is the debate around Ofsted reform?")

QUESTION: What is the debate around Ofsted reform?

[Querying full corpus (with Schools Week)...]
[Querying NM corpus (without Schools Week)...]

FULL CORPUS ANSWER (with Schools Week):
----------------------------------------
Based on the provided articles, the debate around Ofsted reform centers on several key areas of contention:

## Core Reform Proposals and Criticisms

**New Report Card System**: Ofsted is implementing new "report card" inspections to replace single-word judgments. Chief Inspector Sir Martyn Oliver describes these as "revolutionary for parents" and providing more nuance since "schools are too complex for a single word judgment" (Schools Week, 2025-02-03). However, the reforms have been widely criticized for being rushed, with Oliver admitting he "only put out foundations of a plan originally because the sector demanded urgent reform" (Schools Week, 2025-07-03).

## Transparency Concerns

**Lack of Consultation Transparency**: A major point of contention is Ofsted'

In [11]:
# Question 4: topic that disappears without media
q4_full, q4_nm = compare_corpora("What has been happening with teacher strikes and pay disputes?")

QUESTION: What has been happening with teacher strikes and pay disputes?

[Querying full corpus (with Schools Week)...]
[Querying NM corpus (without Schools Week)...]

FULL CORPUS ANSWER (with Schools Week):
----------------------------------------
Based on the articles provided, there has been an extensive and prolonged teacher strike and pay dispute throughout 2023, involving multiple unions and escalating tensions with the government.

## Scale and Timeline of Strikes

The National Education Union (NEU) conducted sustained strike action throughout 2023. By May 2023, the NEU had "held eight days of action in England" with plans for three more after exams (Article 4, May 4, 2023). The strikes began in February 2023, with the NEU initially planning "six days of strike action in England in February and March" (Article 6, January 20, 2023). Additional strikes were announced for July 2023, with the NEU scheduling action for "Wednesday july and Friday july" (Article 3, June 17, 2023).

## 

In [12]:
# Question 5: topic that only appears without media
q5_full, q5_nm = compare_corpora("What regulatory compliance issues are affecting academies?")

QUESTION: What regulatory compliance issues are affecting academies?

[Querying full corpus (with Schools Week)...]
[Querying NM corpus (without Schools Week)...]

FULL CORPUS ANSWER (with Schools Week):
----------------------------------------
Based on the provided articles, several key regulatory compliance issues are affecting academies:

## New Compliance Framework and Enforcement Powers

The government is introducing **new academy compliance orders** to address trusts breaking rules around admissions, uniforms, and parental complaints (Schools Week, 2024-12-19). These orders will enforce requirements including ensuring academies cooperate with councils on admissions, follow the national curriculum, and adhere to proposed uniform caps.

Currently, academies breaching funding agreements can only be issued termination warning notices, but the government recognized that "where non compliance is minor termination may not be a proportionate response" (Schools Week, 2024-12-19).

## Teac

#### Interpretation

The five questions are designed to expose different aspects of specification sensitivity in RAG:

1. **Broad question** — the full corpus answer will be shaped by Schools Week's editorial agenda (teacher pay, Ofsted, RAAC). The NM answer will reflect government/institutional priorities.
2. **AI in schools** — multi-source topic, answers should partially overlap but differ in framing (media: opportunity/risk; government: regulation/standards).
3. **Ofsted reform** — heavily media-driven. The NM corpus may not have enough Ofsted content to give a rich answer.
4. **Teacher strikes** — almost entirely Schools Week content. The NM corpus will struggle to answer this at all, demonstrating that the "knowledge" was in one source.
5. **Regulatory compliance** — invisible in the full corpus (drowned out by media), prominent in the NM corpus (ESFA triplet). The NM answer should be richer.

This is the topic model finding replicated in a RAG pipeline: source composition determines what the system knows, and a user has no way to see this unless it is disclosed.

## 7. Langfuse logging

In [13]:
# Optional: add Langfuse tracing to RAG calls
# Requires LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY in .env

try:
    from langfuse import Langfuse
    
    langfuse = Langfuse(
        public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", ""),
        secret_key=os.environ.get("LANGFUSE_SECRET_KEY", ""),
        host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
    )
    
    def rag_answer_traced(question, index, corpus_df, embed_model, corpus_label, k=10):
        """RAG answer with Langfuse tracing."""
        trace = langfuse.trace(
            name="rag_query",
            metadata={"corpus": corpus_label, "k": k},
        )
        
        # Retrieval span
        retrieval_span = trace.span(name="retrieval")
        results = retrieve(question, index, corpus_df, embed_model, k=k)
        retrieval_span.end(metadata={"n_results": len(results), "sources": [r['source'] for r in results]})
        
        # Build context
        context_parts = []
        for i, r in enumerate(results):
            context_parts.append(
                f"ARTICLE {i+1} (source: {r['source']}, date: {r['date']}, topic: {r['topic']}):\n{r['text']}"
            )
        context = "\n\n".join(context_parts)
        
        # Generation span
        generation = trace.generation(
            name="answer_generation",
            model="claude-sonnet-4-20250514",
            input=question,
        )
        
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            messages=[{"role": "user", "content": f"""You are an education policy analyst. Based on the following articles from the {corpus_label} corpus, answer this question:\n\nQuestion: {question}\n\n{context}\n\nAnswer based only on the articles provided. Cite which sources and dates support your points."""}],
        )
        
        answer = response.content[0].text
        generation.end(output=answer)
        
        langfuse.flush()
        print(f"  Langfuse trace: {trace.id}")
        
        return {
            "answer": answer,
            "sources_used": [r["source"] for r in results],
            "topics_covered": list(set(r["topic"] for r in results)),
            "n_retrieved": len(results),
            "corpus": corpus_label,
            "trace_id": trace.id,
        }
    
    print("Langfuse logging enabled.")
    
except Exception as e:
    print(f"Langfuse not configured ({e}). Using untraced version.")
    rag_answer_traced = rag_answer

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


Langfuse logging enabled.


## 8. Save comparison results

In [14]:
# Save all comparison results for the dashboard
all_comparisons = {
    "questions": [
        {"question": "What are the main issues in English education policy?", "full": q1_full, "nm": q1_nm},
        {"question": "What is the government doing about AI in schools?", "full": q2_full, "nm": q2_nm},
        {"question": "What is the debate around Ofsted reform?", "full": q3_full, "nm": q3_nm},
        {"question": "What has been happening with teacher strikes and pay disputes?", "full": q4_full, "nm": q4_nm},
        {"question": "What regulatory compliance issues are affecting academies?", "full": q5_full, "nm": q5_nm},
    ],
    "embedding_model": "all-MiniLM-L6-v2",
    "llm_model": "claude-sonnet-4-20250514",
    "retrieval_k": 10,
    "full_corpus_size": len(full_df),
    "nm_corpus_size": len(nm_df),
}

with open("/workspaces/AM1_topic_modelling/data/evaluation_outputs/rag_comparison.json", "w") as f:
    json.dump(all_comparisons, f, indent=2)

print("Saved to data/evaluation_outputs/rag_comparison.json")

Saved to data/evaluation_outputs/rag_comparison.json


## Conclusions

The RAG comparison demonstrates specification sensitivity in the architecture that matters in 2026. The same question asked of two corpora — one with media, one without — produces different answers because the retrieval step returns different documents. The user sees a confident answer and has no way to know that a different corpus would have produced a different one.

**Key findings to look for in the results:**
- Q1 (broad): full corpus answer shaped by media editorial agenda; NM answer shaped by government/institutional priorities
- Q4 (teacher strikes): full corpus gives a rich answer; NM corpus struggles because the knowledge was in one source
- Q5 (regulatory compliance): NM corpus gives a richer answer; full corpus doesn't surface ESFA content because media volume drowns it out

**Specification choices in this RAG pipeline:**
- Embedding model: all-MiniLM-L6-v2 (a different model would retrieve different articles)
- Retrieval k: 10 (more or fewer would change the answer)
- Corpus: full vs NM (the core demonstration)
- LLM: Claude Sonnet (a different model would synthesise differently)
- Prompt: the system prompt shapes how the LLM uses the retrieved context

Every one of these is a specification choice. None is visible to the end user. This is specification sensitivity all the way down.